In [14]:
# --- Scrap notebook: gather seasons & plot per-year "always bet home" profit ---
from pathlib import Path
import io, sys, time, math, textwrap, requests
import pandas as pd
import matplotlib.pyplot as plt

# ----------------- config -----------------
LEAGUE = "E0"                 # EPL; use E1 (Champ), E2, etc., if needed
STAKE = 100                   # $ per bet
OUTDIR = Path("outputs/figs")
DATADIR = Path("data/raw")
DATADIR.mkdir(parents=True, exist_ok=True)
OUTDIR.mkdir(parents=True, exist_ok=True)

def season_codes(start=2003, end=2025):
    """Return list like ['0304','0405',...,'2526'] inclusive."""
    codes = []
    for y in range(start, end+1):
        a = str(y)[-2:]
        b = str((y+1))[-2:]
        codes.append(f"{a}{b}")
    return codes

def fetch_csv(season_code, league=LEAGUE, destdir=DATADIR, sleep=0.6):
    """Download season CSV if missing; return local path or None if 404."""
    url = f"https://www.football-data.co.uk/mmz4281/{season_code}/{league}.csv"
    path = destdir / f"{league}_{season_code}.csv"
    if path.exists() and path.stat().st_size > 0:
        return path
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200 and len(r.content) > 100:
            path.write_bytes(r.content)
            time.sleep(sleep)
            return path
        else:
            print(f"[skip] {season_code} {league}: {r.status_code}")
            return None
    except Exception as e:
        print(f"[err] {season_code} {league}: {e}")
        return None

def read_tolerant(path):
    """Robust CSV reader (handles occasional malformed rows)."""
    try:
        return pd.read_csv(path, dayfirst=True, encoding="latin-1")
    except pd.errors.ParserError:
        return pd.read_csv(path, engine="python", dayfirst=True,
                           on_bad_lines="skip", encoding="latin-1")

def pick_odds_cols(df):
    """
    Prefer closing Bet365 (B365CH/B365CD/B365CA). Fallbacks:
    Pinnacle closing (PSCH/PSCD/PSCA) → market Avg/Max closing (AvgCH/MaxCH...)
    → pre-closing Bet365 (B365H/D/A) as last resort.
    Returns tuple of column names (H, D, A) present in df or (None, None, None).
    """
    pref_sets = [
        ("B365CH","B365CD","B365CA"),
        ("PSCH","PSCD","PSCA"),
        ("AvgCH","AvgCD","AvgCA"),
        ("MaxCH","MaxCD","MaxCA"),
        ("B365H","B365D","B365A"),
        ("PSH","PSD","PSA"),
        ("AvgH","AvgD","AvgA"),
        ("MaxH","MaxD","MaxA"),
    ]
    for h,d,a in pref_sets:
        if all(c in df.columns for c in (h,d,a)):
            return h,d,a
    return (None, None, None)

def clean_and_order(df):
    # Parse date/time; sort by date to make the cumulative path sensible
    # Football-Data "Date" is dd/mm/yy for most seasons.
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
        df = df.sort_values("Date")
    return df

def compute_home_cumprofit(df, stake=STAKE):
    """Return series of cumulative profit for always betting home at given stake."""
    h_col, d_col, a_col = pick_odds_cols(df)
    if h_col is None:
        return None, None, "No suitable odds columns found"

    # Ensure numeric
    for c in (h_col, d_col, a_col):
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Consider only rows with a valid price and full-time result
    if "FTR" not in df.columns:
        return None, None, "Missing FTR (full-time result) column"
    sub = df.loc[df[h_col].notna() & df["FTR"].notna()].copy()

    # Realized profit per match for betting on the HOME outcome
    win_mask = (sub["FTR"].astype(str).str.upper() == "H")
    # Payout on win = (odds - 1) * stake; loss = -stake
    sub["ret_home"] = win_mask.astype(int) * ((sub[h_col] - 1.0) * stake) + (~win_mask).astype(int) * (-stake)
    sub["cum_profit"] = sub["ret_home"].cumsum()
    label = f"Always Home @ {h_col}"
    return sub, label, None

def plot_cumulative(series_df, label, season_code, league=LEAGUE, outdir=OUTDIR):
    if series_df is None or series_df.empty:
        return
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(range(1, len(series_df)+1), series_df["cum_profit"])
    ax.axhline(0, linewidth=1)
    ax.set_title(f"{league} {season_code}: Cumulative P&L ($100 stake) — {label}")
    ax.set_xlabel("Matches (chronological)")
    ax.set_ylabel("Cumulative profit ($)")
    fig.tight_layout()
    outfile = outdir / f"{league}_{season_code}_home_bet_cumprofit.png"
    fig.savefig(outfile, dpi=150)
    plt.close(fig)

# ----------------- run end-to-end -----------------
summary = []

for code in season_codes(2003, 2025):   # 2003/04 → 2025/26
    p = fetch_csv(code)
    if not p: 
        continue
    df = read_tolerant(p)
    df = clean_and_order(df)
    series_df, label, err = compute_home_cumprofit(df)
    if err:
        print(f"[warn] {p.name}: {err}")
        continue
    plot_cumulative(series_df, label, code)
    final_pnl = float(series_df["cum_profit"].iloc[-1]) if len(series_df) else math.nan
    bets = int(series_df.shape[0])
    summary.append({"season": code, "bets": bets, "final_PnL_$": final_pnl, "odds_used": label.split("@")[-1].strip()})

# Save a small season-level summary CSV
pd.DataFrame(summary).to_csv(OUTDIR.parent / "home_bet_summary.csv", index=False)

print(f"Done. Figures in {OUTDIR.resolve()} and summary CSV at { (OUTDIR.parent / 'home_bet_summary.csv').resolve() }")

/tmp/ipykernel_270247/1225944549.py:77: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
/tmp/ipykernel_270247/1225944549.py:77: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
/tmp/ipykernel_270247/1225944549.py:77: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
/tmp/ipykernel_270247/1225944549.py:77: UserWarning: Could not infer format, so each element will be parse

Done. Figures in /home/pangea/Documents/ninaad/football-betting/notebooks/outputs/figs and summary CSV at /home/pangea/Documents/ninaad/football-betting/notebooks/outputs/home_bet_summary.csv
